# Task 4: Context-Aware RAG Chatbot Using LangChain
## DevelopersHub Corporation — AI/ML Engineering Internship

**Objective:**

A chatbot that:
1. Reads a set of documents and stores them in a **vector database** (FAISS)
2. When you ask a question, it **retrieves the most relevant chunks** from those documents
3. Feeds the retrieved chunks + your question into an **LLM** to generate an answer
4. **Remembers the conversation history** so follow-up questions work correctly

This is called **RAG — Retrieval-Augmented Generation**.

## Step 1: Importing necessary libraries

In [1]:
from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

print("Libraries imported successfully!")

c:\Users\Muhammad Noman\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


## Step 2: Preparing the KNOWLEDGE BASE

In [4]:
# 1. Load PDF
loader = PyPDFLoader("data/book.pdf")
data = loader.load()

# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

# 3. Create Embeddings (The math part)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 4. Save locally
vector_store = FAISS.from_documents(docs, embeddings)
vector_store.save_local("faiss_index")

print("✅ Step 2 Complete: 'faiss_index' folder created!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5579.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Step 2 Complete: 'faiss_index' folder created!


## Step 3: Connecting Hugging Face to the vector store

In [5]:
def get_chat_chain(vector_store):
    llm = HuggingFaceEndpoint(
        repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
        huggingfacehub_api_token="..."
    )

    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vector_store.as_retriever(),
        memory=memory
    )
    return chain